# Notebook 02 - Adapt Dataset With Adaption

This notebook follows the Adaption SDK flow from the docs: install/import the SDK, load credentials, upload the prepared file, confirm column mapping, run an estimate, adapt the rows that Adaption accepted during upload, poll for completion, download the adapted output, and inspect before/after rows.

**Files read:**
- [`../configs/adaption_run.yaml`](../configs/adaption_run.yaml) - paths, column mapping, dynamic job-size behavior, and timeout settings.
- [`../configs/adaption_blueprint.yaml`](../configs/adaption_blueprint.yaml) - qualitative instructions and Adaption recipe settings.
- [`../data/processed/kakugo_adaption_input.csv`](../data/processed/kakugo_adaption_input.csv) - prepared prompt/response rows from Notebook 01.

**Files written:**
- [`../data/adapted/kakugo_adapted.csv`](../data/adapted/kakugo_adapted.csv) - adapted dataset downloaded from Adaption.
- [`../data/processed/kakugo_adapted_sft.jsonl`](../data/processed/kakugo_adapted_sft.jsonl) - adapted data converted to chat-format JSONL for SFT.
- [`../results/adaption_run_metadata.json`](../results/adaption_run_metadata.json) - run metadata, including the uploaded dataset ID and run plan.
- [`../results/adaption_dataset_diagnosis.json`](../results/adaption_dataset_diagnosis.json) - API-visible dataset description, status, and evaluation metadata captured before the adaptation run.
- [`../results/adaption_dataset_evaluation.json`](../results/adaption_dataset_evaluation.json) - API-visible quality/evaluation metadata captured after the adaptation run.


In [1]:
import json
import os
import sys
from pathlib import Path
import pandas as pd

from adaption import Adaption

In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )

ROOT = find_repo_root(Path.cwd())
SRC = str(ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
ROOT_STR = str(ROOT)
if ROOT_STR not in sys.path:
    sys.path.insert(0, ROOT_STR)

In [3]:
from kirundi_sft_starter.adaption import (
    blueprint_text,
    capture_dataset_diagnosis,
    download_to_file,
    get_adaptation_job_config,
    to_plain_data,
    wait_for_completion,
    wait_until_ingested,
)
from kirundi_sft_starter.data import convert_adapted_to_sft
from kirundi_sft_starter.utils import ensure_dir, load_env, load_yaml

load_env()

In [4]:
project_config = load_yaml(ROOT / 'configs/project.yaml')
adaption_run_config = load_yaml(ROOT / 'configs/adaption_run.yaml')
adaptation_job_config = get_adaptation_job_config(adaption_run_config)

blueprint_config = load_yaml(ROOT / 'configs/adaption_blueprint.yaml')

if not os.environ.get('ADAPTION_API_KEY'):
    raise RuntimeError('Missing ADAPTION_API_KEY. Add it to .env before running this notebook.')

## Inspect the input and column mapping

Adaption needs to know which columns contain the prompt and completion. This repo maps `instruction` to `prompt` and `response` to `completion`.

In [5]:
adaption_input_path = ROOT / adaption_run_config['input_path']

adaption_input_df = pd.read_csv(adaption_input_path)

print(f"Size: {adaption_input_df.shape[0]}")

adaption_input_df.head()

Size: 200


,example_id,instruction,response
0,kakugo_00000,Amahitamo aboneka:\n* bibi\n* vyiza\nIncamake ...,bibi
1,kakugo_00001,Kuri igiciro ki igitabo gikoresheje Rs. 150 ki...,"Ukwambere, reka tubare igiciro co kugurisha ki..."
2,kakugo_00002,Ndashaka kumenya uburyo nabona ibikoresho bira...,"Ndagutahura umwuga w’ugura ibikoresho biramba,..."
3,kakugo_00003,Ndakenera kumenya neza ahantu hiyobora kwandik...,Ndagushigikira ko ntashobora gutanga ivyo vyan...
4,kakugo_00004,Hari uburyo bwo kubona cookie runaka hakoreshe...,"Yego! Muri Go, ushobora gukoresha imikorere ya..."


In [6]:
adaption_run_config['column_mapping']

{'prompt': 'instruction', 'completion': 'response'}

In [7]:
print(f"Path: ROOT/{'configs/adaption_blueprint.yaml'}\n")
      
brand_controls = dict(blueprint_config['adaption_brand_controls'])

brand_controls['blueprint'] = blueprint_text(blueprint_config)

print(brand_controls['blueprint'][:1000])

Path: ROOT/configs/adaption_blueprint.yaml

Goal:
Improve this dataset for supervised fine-tuning a small assistant that can answer beginner-friendly questions in Kirundi. The input dataset may contain Kinyarwanda-like or mixed Kinyarwanda/Kirundi examples, so part of the task is language normalization into Kirundi.

Language policy:
- Target language: Kirundi
- Source language issue: Some rows may be mislabeled, mixed, or written in Kinyarwanda-like language rather than clean Kirundi.
- Treat Kirundi as the target language for both the instruction and response.
- If a row is Kinyarwanda-like or mixed but the meaning is clear, rewrite it into natural Kirundi while preserving the original task and answer.
- If the task explicitly asks for another language, preserve that requested output language.
- If local entities, places, currencies, or institutions appear because the row is Kinyarwanda-specific, do not invent Burundi-specific replacements. Prefer neutral wording unless those details

## Upload the dataset



In [8]:
client = Adaption(api_key=os.environ['ADAPTION_API_KEY'])
upload = client.datasets.upload_file(str(adaption_input_path), name=adaption_run_config['dataset_name'])
dataset_id = upload.dataset_id
print('Dataset ID:', dataset_id)

ingested_status = wait_until_ingested(
    client,
    dataset_id,
    timeout_seconds=adaption_run_config.get('ingestion_timeout_seconds'),
)
ingested_status_data = to_plain_data(ingested_status)
print('Dataset ingestion metadata:', ingested_status_data)

api_row_count = ingested_status_data.get('row_count')
if api_row_count is None:
    raise RuntimeError('Adaption did not report a row_count for the uploaded dataset.')

accepted_rows = int(api_row_count)
if accepted_rows <= 0:
    raise RuntimeError(f'Adaption reported row_count={accepted_rows}; cannot launch an adaptation run.')

configured_max_rows = adaptation_job_config.get('max_rows')
if configured_max_rows is None:
    requested_rows = accepted_rows
else:
    requested_rows = min(int(configured_max_rows), accepted_rows)

adaptation_job_config = dict(adaptation_job_config)
adaptation_job_config['max_rows'] = requested_rows

print(
    f"Local CSV rows: {len(adaption_input_df)} | "
    f"Adaption accepted rows: {accepted_rows} | "
    f"Rows requested for adaptation: {requested_rows}"
)

Dataset ID: 7aa58cf2-5576-430c-bd4a-66b0345cb836
Dataset ingestion metadata: {'dataset_id': '7aa58cf2-5576-430c-bd4a-66b0345cb836', 'row_count': 182, 'status': 'pending'}
Local CSV rows: 200 | Adaption accepted rows: 182 | Rows requested for adaptation: 182


## Capture API-visible diagnosis metadata

The Adaption UI may show a richer **Data Diagnosis** page before launch. The public API docs expose the closest quick metadata through dataset get/status/list calls. This cell intentionally does not call `get_evaluation` before the adaptation run because that endpoint can block before evaluation output exists.

In [9]:
diagnosis_path = ROOT / adaption_run_config.get(
    'diagnosis_path',
    'results/adaption_dataset_diagnosis.json',
)
ensure_dir(diagnosis_path)

diagnosis_metadata = capture_dataset_diagnosis(
    client,
    dataset_id,
    adaption_run_config['dataset_name'],
)
diagnosis_path.write_text(
    json.dumps(diagnosis_metadata, indent=2, ensure_ascii=False, default=str),
    encoding='utf-8',
)

dataset_record = diagnosis_metadata.get('dataset_record') or {}
dataset_status = diagnosis_metadata.get('dataset_status') or {}
listed_dataset = diagnosis_metadata.get('listed_dataset') or {}

summary = {
    'dataset_id': dataset_id,
    'name': listed_dataset.get('name') or dataset_record.get('name'),
    'description': listed_dataset.get('description') or dataset_record.get('description'),
    'status': dataset_status.get('status') or listed_dataset.get('status'),
    'row_count': dataset_status.get('row_count') or listed_dataset.get('row_count'),
    'evaluation': 'retrieved after adaptation run',
    'saved_to': str(diagnosis_path.relative_to(ROOT)),
}
pd.DataFrame([summary]).T.rename(columns={0: 'value'})

,value
dataset_id,7aa58cf2-5576-430c-bd4a-66b0345cb836
name,kakugo-run-kirundi-adaption-run
description,None
status,pending
row_count,182
evaluation,retrieved after adaptation run
saved_to,results/adaption_dataset_diagnosis.json


## Run estimate

This cell asks Adaption for an estimate before launching the actual adaptation run. The estimate uses the row count that Adaption accepted during upload.

In [10]:
estimate = client.datasets.run(
    dataset_id,
    column_mapping=adaption_run_config['column_mapping'],
    brand_controls=brand_controls,
    recipe_specification=blueprint_config['adaption_recipe_specification'],
    job_specification={'max_rows': adaptation_job_config['max_rows']},
    estimate=True,
)
print('Estimated credits:', estimate.estimated_credits_consumed)

Estimated credits: 2.0


## Run adaptation job and save outputs

This starts the configured adaptation job, waits for completion, downloads the adapted CSV, and writes run metadata locally. The requested row count comes from Adaption's accepted upload row count unless `configs/adaption_run.yaml` sets an explicit smaller cap. In `configs/adaption_run.yaml`, `timeout_seconds: null` means keep waiting instead of failing after a fixed time. The wait cell prints a heartbeat every 60 seconds.

In [11]:
adaption_job = client.datasets.run(
    dataset_id,
    column_mapping=adaption_run_config['column_mapping'],
    brand_controls=brand_controls,
    recipe_specification=blueprint_config['adaption_recipe_specification'],
    job_specification={'max_rows': adaptation_job_config['max_rows']},
)
print('Adaption run started:', adaption_job.run_id)

Adaption run started: dataset-7aa58cf2-5576-430c-bd4a-66b0345cb836-1777847641397


In [12]:
final = wait_for_completion(
    client,
    dataset_id,
    timeout_seconds=adaptation_job_config['timeout_seconds'],
    heartbeat_seconds=60,
)
print('Adaption run finished:', final.status)
if getattr(final, 'error', None):
    raise RuntimeError(final.error.message)

Waiting for Adaption run to finish...
Adaption run finished: succeeded


## Retrieve Adaption dataset and evaluation metadata


In [13]:
evaluation_path = ROOT / adaption_run_config.get(
    'evaluation_path',
    'results/adaption_dataset_evaluation.json',
)
ensure_dir(evaluation_path)

post_run_evaluation = client.datasets.get_evaluation(dataset_id)
post_run_evaluation_data = to_plain_data(post_run_evaluation)
evaluation_path.write_text(
    json.dumps(post_run_evaluation_data, indent=2, ensure_ascii=False, default=str),
    encoding='utf-8',
)

def find_nested_value(value, key):
    if isinstance(value, dict):
        if key in value:
            return value[key]
        for item in value.values():
            found = find_nested_value(item, key)
            if found is not None:
                return found
    elif isinstance(value, list):
        for item in value:
            found = find_nested_value(item, key)
            if found is not None:
                return found
    return None

In [14]:
evaluation_summary = {
    'grade_before': find_nested_value(post_run_evaluation_data, 'grade_before'),
    'grade_after': find_nested_value(post_run_evaluation_data, 'grade_after'),
    'score_before': find_nested_value(post_run_evaluation_data, 'score_before'),
    'score_after': find_nested_value(post_run_evaluation_data, 'score_after'),
    'status': find_nested_value(post_run_evaluation_data, 'status'),
    'saved_to': str(evaluation_path.relative_to(ROOT)),
}
pd.DataFrame([evaluation_summary]).T.rename(columns={0: 'value'})

,value
grade_before,C
grade_after,B
score_before,6.0
score_after,7.9
status,pending
saved_to,results/adaption_dataset_evaluation.json


In [15]:
adapted_output_path = ROOT / adaption_run_config['output_path']
metadata_path = ROOT / adaption_run_config['metadata_path']

download_to_file(client, dataset_id, adapted_output_path)

ensure_dir(metadata_path)
metadata = {
    'dataset_id': dataset_id,
    'adaption_run_id': adaption_job.run_id,
    'input_path': str(adaption_input_path.relative_to(ROOT)),
    'output_path': str(adapted_output_path.relative_to(ROOT)),
    'diagnosis_path': str(diagnosis_path.relative_to(ROOT)),
    'evaluation_path': str(evaluation_path.relative_to(ROOT)) if 'evaluation_path' in globals() else None,
    'column_mapping': adaption_run_config['column_mapping'],
    'adaptation_job': adaptation_job_config,
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Adapted CSV:', adapted_output_path)
print('Run metadata:', metadata_path)

Adapted CSV: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/data/adapted/kakugo_adapted.csv
Run metadata: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/results/adaption_run_metadata.json


In [16]:
adapted_df = pd.read_csv(adapted_output_path)
print('Adapted rows:', len(adapted_df))
adapted_df.head()

Adapted rows: 182


,instruction,response,enhanced_prompt,enhanced_completion,example_id,prompt_safety_issues,response_safety_issues
0,Umurepublicain akomeye afise ubwoba ku ntwaro ...,Kongera umwimbu w’amakara no gufungura imihora...,# Incamake y'Inkuru\nUmutegetsi akomeye mu mug...,Kwongera umwimbu w'amakara no gufungura imihor...,kakugo_00028,NaN,NaN
1,Bikajijwe ko igiteranyo c'ibibare bibiri ari 2...,"Fata imibare ibiri $x$ na $y$, aho $x>y$. Tura...",Umuti w'ikibazo cya matematike:\n\nAmakuru yat...,"Ehe uko wabona umubare munini, dukurikije inta...",kakugo_00112,NaN,NaN
2,Ndambiriye kumenya ukuri kw'ikintu kiguma ku m...,Umunyeshuri w’amahembe y’ubutaka (geoloji) niw...,Ndambiriye kumenya ukuri kw'ikintu kiguma ku m...,Umuhanga mu bumenyi bw'ubutaka (geologiya) ni ...,kakugo_00111,NaN,NaN
3,Ndashaka urutonde rw'inkuru za politiki za ejo...,"Ndababajwe, sinshobora kubona inkuru za politi...",Ndashaka urutonde rw'inkuru za politiki zivuga...,Ntabwo dufise ubushobozi bwo gutanga inkuru zi...,kakugo_00153,NaN,NaN
4,Ndashaka kumenya izina ry'indirimbo igezweho i...,"Ndagukorera neza, ndashaka kumenya andi makuru...",Ndashaka kumenya izina ry'indirimbo igezweho i...,Muraho neza! Hari indirimbo nyinshi zigezweho ...,kakugo_00018,NaN,NaN


In [17]:
metadata_path = ROOT / adaption_run_config['metadata_path']

with metadata_path.open('r', encoding='utf-8') as f:
    adaption_run_metadata = json.load(f)

pd.DataFrame([adaption_run_metadata]).T.rename(columns={0: 'value'})

,value
dataset_id,7aa58cf2-5576-430c-bd4a-66b0345cb836
adaption_run_id,dataset-7aa58cf2-5576-430c-bd4a-66b0345cb836-1...
input_path,data/processed/kakugo_adaption_input.csv
output_path,data/adapted/kakugo_adapted.csv
diagnosis_path,results/adaption_dataset_diagnosis.json
evaluation_path,results/adaption_dataset_evaluation.json
column_mapping,"{'prompt': 'instruction', 'completion': 'respo..."
adaptation_job,"{'max_rows': 182, 'timeout_seconds': None}"


## Convert for SFT

Convert the adapted CSV into the same chat-format JSONL structure used by the raw SFT run.

In [18]:
adapted_sft_path = ROOT / project_config['datasets']['sft']['adapted_sft_path']
adapted_sft_df = convert_adapted_to_sft(adapted_output_path, adapted_sft_path)

print('Converted adapted examples:', len(adapted_sft_df))
print('Adapted SFT JSONL:', adapted_sft_path)

Converted adapted examples: 182
Adapted SFT JSONL: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/data/processed/kakugo_adapted_sft.jsonl


In [19]:
with adapted_sft_path.open('r', encoding='utf-8') as f:
    adapted_sft_preview_rows = [json.loads(line) for _, line in zip(range(3), f)]

pd.DataFrame(
    [
        {
            'example_id': row.get('metadata', {}).get('example_id'),
            'user': row['messages'][0]['content'],
            'assistant': row['messages'][1]['content'],
        }
        for row in adapted_sft_preview_rows
    ]
)

,example_id,user,assistant
0,kakugo_00028,Umurepublicain akomeye afise ubwoba ku ntwaro ...,Kongera umwimbu w’amakara no gufungura imihora...
1,kakugo_00112,Bikajijwe ko igiteranyo c'ibibare bibiri ari 2...,"Fata imibare ibiri $x$ na $y$, aho $x>y$. Tura..."
2,kakugo_00111,Ndambiriye kumenya ukuri kw'ikintu kiguma ku m...,Umunyeshuri w’amahembe y’ubutaka (geoloji) niw...
